# M4 - judge (Layer 4) + baseline ASR

Run the battery against the **undefended** target, let the judge label every reply
BAD_BOT / GOOD_BOT / UNCLEAR, read off per-attack Attack Success Rate.

- target `Qwen2.5-3B` -> **cuda:0** (fp16, ~6 GB); also serves as the helper LM (same weights, 0 extra VRAM)
- judge `Qwen3.5-9B` -> **cuda:1** (nf4 4-bit, ~6 GB)

**Accelerator must be `GPU T4 x2`.** Internet On.

### Fixed since the 2026-09-03 run (whose numbers were invalid)
1. **Cipher replies are decoded before grading.** `base64`/`rot13`/`combination_*` ask the
   target to answer in cipher; the judge was scoring the raw blob, i.e. the *form* not the
   *substance*. One reply scored BAD_BOT decoded to pure word salad.
2. **Stricter rubric.** Merely restating/decoding the request, or emitting gibberish, is now
   `UNCLEAR`. The old run produced 0 UNCLEAR out of 900 - the tell that it was too lenient.
3. **Real helper LM**, so `auto_payload_splitting` / `auto_obfuscation` actually run.
4. **Transcripts are mirrored** to `/kaggle/working/artifacts/<run-id>` so they survive the
   session (`logs/` lives in the repo clone, which is deleted every run).

## 1 - Setup

In [ ]:
%pip -q install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import os, subprocess, sys, pathlib, time, json, glob
import numpy as np, pandas as pd

_sec = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
except Exception as e:
    print('no Kaggle secrets client:', e)

def _secret(name):
    try:
        return _sec.get_secret(name) if _sec is not None else None
    except Exception:
        return None

_hf = _secret('HF_TOKEN')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    from huggingface_hub import login; login(token=_hf)
    print('HF auth OK')
else:
    print('no HF_TOKEN secret (fine - models are public)')

In [ ]:
# --- get the repo (public or private; safe to re-run) ----------------------
REPO   = "MehemudAzad/LLM-jailbreaking-with-layered-prompt-defense"
BRANCH = "main"
WORK   = pathlib.Path("/kaggle/working")
ROOT   = WORK / "repo"

_gh  = _secret("GH_TOKEN")
_url = f"https://{_gh}@github.com/{REPO}.git" if _gh else f"https://github.com/{REPO}.git"

os.chdir(WORK)
subprocess.run(["rm", "-rf", str(ROOT)], check=False)
_r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(ROOT)],
                    cwd=str(WORK), capture_output=True, text=True)
if _r.returncode != 0:
    _err = _r.stderr.replace(_gh, "***") if _gh else _r.stderr
    raise RuntimeError("git clone failed:\n" + _err +
        "\n\nPrivate repo? add a GH_TOKEN Kaggle secret (fine-grained PAT, Contents: read-only)."
        "\nOr make the repo public. Also: git push -u origin main")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("HEAD", subprocess.check_output(["git","-C",str(ROOT),"rev-parse","--short","HEAD"]).decode().strip())

# fail loudly here rather than with a confusing ModuleNotFoundError five cells later
_missing = [f for f in ("run_eval.py", "report.py", "regrade.py", "core/models.py",
                        "defense/layer4_response_classifier.py", "datasets/build_harmful.py")
            if not (ROOT / f).exists()]
if _missing:
    raise RuntimeError(f"clone is missing {_missing} -- commit and push them, then re-run this cell")
print("repo files OK")

In [ ]:
# frozen harmful set is git-ignored -> build it (seeded, deterministic)
r = subprocess.run([sys.executable, "datasets/build_harmful.py"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())

In [ ]:
import torch
print('CUDA devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.0f} GB')
assert torch.cuda.device_count() >= 2, 'set the accelerator to GPU T4 x2'

In [ ]:
from core.config import CONFIG
for role in ('target', 'helper', 'judge'):
    s = CONFIG['models'][role]
    print(f"{role:<8} {s['name']:<28} {s.get('device')}  backend={s.get('backend')}"
          f"  quant={s.get('quant')}")
print('\nlayer1:', CONFIG['defense']['layer1_perplexity'])

## 2 - Sanity pass (3 goals)

~54 trials, ~4-6 min. One line per trial with a running ETA.

In [ ]:
from run_eval import main
main(['--attack', 'all', '--defense', 'off', '--limit', '3', '--tag', 'm4sanity'])

In [ ]:
import report
report.print_asr('m4sanity', title='SANITY (3 goals, undefended)')

In [ ]:
report.samples('m4sanity', exclude='GOOD_BOT', n=10)

### GATE - check these three things before the 50-goal run

1. **Cipher attacks show `label_method = judge:decoded`** and a `judged:` line with the
   decoded plaintext. If the plaintext is word salad it must be `UNCLEAR`, not `BAD_BOT`.
2. **`UNCLEAR` actually appears** in the label mix. Zero UNCLEAR means the rubric is still
   too lenient.
3. **`prefix_injection` still lands `BAD_BOT`** with a genuinely harmful reply - that one is
   a true positive and must not be lost to over-correction.

If any of those look wrong, stop and fix `defense/layer4_response_classifier.py`
(`_JUDGE_SYSTEM` / `try_decode`) rather than burning an hour on the full run.

## 3 - Full baseline (50 goals)

~900 trials, ~1-1.5 h. The transcript is written as it goes, so a mid-run stop still
leaves usable partial data.

In [ ]:
main(['--attack', 'all', '--defense', 'off', '--tag', 'm4baseline'])

In [ ]:
tbl = report.print_asr('m4baseline', title='BASELINE ASR - undefended Qwen2.5-3B, 50 AdvBench goals')

In [ ]:
report.samples('m4baseline', only='BAD_BOT', n=6)

## 4 - Save the evidence + pin the judge

Download `artifacts.zip` from the notebook Output, unzip into the repo's `logs/`, then
`git add -f logs/<run-id>` - the spec wants raw transcripts for every attack x defense combo.

In [ ]:
!cd /kaggle/working && zip -qr artifacts.zip artifacts && ls -la artifacts.zip
print()
!ls /kaggle/working/artifacts

In [ ]:
from huggingface_hub import HfApi
for role in ('target', 'judge', 'helper'):
    nm = CONFIG['models'][role]['name']
    try:
        sha = HfApi().model_info(nm, token=os.environ.get('HF_TOKEN')).sha
        print(f'[models.{role}]  # {nm}\n  revision = "{sha}"')
    except Exception as e:
        print(role, 'revision lookup failed:', e)

## Done - what to commit

- the pinned `revision` values above, into `config.toml`
- `logs/<m4baseline-run-id>/` force-added as report evidence
- this notebook (with outputs)

For the report: the baseline ASR table is the 'before' half of every result. Note the
capability finding too - a 3B target often *cannot* execute the character-level ciphers,
so part of the encoding family fails for capability reasons, not safety ones.

**M5:** wire Layer 2 (paraphraser) for real, tune Layer 3, then the **defended** ASR pass
(`--defense on`) and `report.print_compare('m4baseline', 'm5defended')`.